In [ ]:
# ============================================================================
# Imports
# ============================================================================

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from hermes.config import GRAPH_DIR
from hermes.graph.io import load_graphml

In [ ]:
# ============================================================================
# Load graph
# ============================================================================

graph = load_graphml(
    GRAPH_DIR / "municipality.graphml"
)

graph

In [ ]:
# ============================================================================
# Graph summary
# ============================================================================

summary = pd.Series(
    {
        "nodes": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
        "self_loops": nx.number_of_selfloops(graph),
        "isolated_nodes": nx.number_of_isolates(graph),
        "weakly_connected_components": (
            nx.number_weakly_connected_components(graph)
        ),
    }
)

summary

In [ ]:
# ============================================================================
# Build mobility flows DataFrame
# ============================================================================

edges_df = nx.to_pandas_edgelist(
    graph,
    source="origin_insee_code",
    target="destination_insee_code",
)

edges_df.head()

In [ ]:
# ============================================================================
# Commuting flow statistics
# ============================================================================

edges_df["commuters"].describe()

In [ ]:
# ============================================================================
# Separate internal and inter-municipal flows
# ============================================================================

internal_flows = edges_df[
    edges_df["origin_insee_code"]
    == edges_df["destination_insee_code"]
].copy()

external_flows = edges_df[
    edges_df["origin_insee_code"]
    != edges_df["destination_insee_code"]
].copy()

print(
    f"Internal flows: {len(internal_flows):,}"
)

print(
    f"Inter-municipal flows: {len(external_flows):,}"
)

In [ ]:
# ============================================================================
# Compare internal and inter-municipal mobility
# ============================================================================

total_commuters = edges_df["commuters"].sum()

internal_commuters = internal_flows["commuters"].sum()
external_commuters = external_flows["commuters"].sum()

mobility_summary = pd.Series(
    {
        "total_commuters": total_commuters,
        "internal_commuters": internal_commuters,
        "inter_municipal_commuters": external_commuters,
        "internal_share": internal_commuters / total_commuters,
        "inter_municipal_share": external_commuters / total_commuters,
    }
)

mobility_summary

In [ ]:
# ============================================================================
# Add municipality names to commuting flows
# ============================================================================

municipality_names = {
    node: attributes.get("municipality")
    for node, attributes in graph.nodes(data=True)
}

external_flows["origin_name"] = (
    external_flows["origin_insee_code"]
    .map(municipality_names)
)

external_flows["destination_name"] = (
    external_flows["destination_insee_code"]
    .map(municipality_names)
)

external_flows.head()

In [ ]:
# ============================================================================
# Largest inter-municipal commuting flows
# ============================================================================

largest_external_flows = (
    external_flows[
        [
            "origin_insee_code",
            "origin_name",
            "destination_insee_code",
            "destination_name",
            "commuters",
        ]
    ]
    .sort_values(
        "commuters",
        ascending=False,
    )
    .head(20)
    .reset_index(drop=True)
)

largest_external_flows

In [ ]:
# ============================================================================
# Compute municipality-level mobility metrics
# ============================================================================

node_metrics = []

for node, attributes in graph.nodes(data=True):

    internal_flow = (
        graph[node][node]["commuters"]
        if graph.has_edge(node, node)
        else 0.0
    )

    outgoing_flow = sum(
        data["commuters"]
        for _, destination, data in graph.out_edges(
            node,
            data=True,
        )
        if destination != node
    )

    incoming_flow = sum(
        data["commuters"]
        for origin, _, data in graph.in_edges(
            node,
            data=True,
        )
        if origin != node
    )

    out_degree = sum(
        1
        for _, destination in graph.out_edges(node)
        if destination != node
    )

    in_degree = sum(
        1
        for origin, _ in graph.in_edges(node)
        if origin != node
    )

    node_metrics.append(
        {
            "insee_code": node,
            "municipality": attributes.get("municipality"),
            "internal_flow": internal_flow,
            "outgoing_flow": outgoing_flow,
            "incoming_flow": incoming_flow,
            "out_degree": out_degree,
            "in_degree": in_degree,
        }
    )

In [ ]:
# Create mobility metrics

mobility_metrics = pd.DataFrame(node_metrics)

mobility_metrics.head()

In [ ]:
# Statistics for mobility metrics 

mobility_metrics[
    [
        "internal_flow",
        "outgoing_flow",
        "incoming_flow",
        "out_degree",
        "in_degree",
    ]
].describe()

In [ ]:
# ============================================================================
# Compute derived mobility indicators
# ============================================================================

mobility_metrics["resident_workers"] = (
    mobility_metrics["internal_flow"]
    + mobility_metrics["outgoing_flow"]
)

mobility_metrics["workplace_workers"] = (
    mobility_metrics["internal_flow"]
    + mobility_metrics["incoming_flow"]
)

mobility_metrics["retention_rate"] = (
    mobility_metrics["internal_flow"]
    / mobility_metrics["resident_workers"]
)

mobility_metrics["out_commuting_rate"] = (
    mobility_metrics["outgoing_flow"]
    / mobility_metrics["resident_workers"]
)

mobility_metrics["net_commuting"] = (
    mobility_metrics["incoming_flow"]
    - mobility_metrics["outgoing_flow"]
)

mobility_metrics["jobs_workers_ratio"] = (
    mobility_metrics["workplace_workers"]
    / mobility_metrics["resident_workers"]
)

In [ ]:
# Clean ratios

ratio_columns = [
    "retention_rate",
    "out_commuting_rate",
    "jobs_workers_ratio",
]

mobility_metrics[ratio_columns] = (
    mobility_metrics[ratio_columns]
    .replace([np.inf, -np.inf], np.nan)
)

In [ ]:
# Stats for derived indicators

mobility_metrics[
    [
        "resident_workers",
        "workplace_workers",
        "retention_rate",
        "out_commuting_rate",
        "net_commuting",
        "jobs_workers_ratio",
    ]
].describe()

In [ ]:
# Sanity check

(
    mobility_metrics["retention_rate"]
    + mobility_metrics["out_commuting_rate"]
).describe()

In [ ]:
# ============================================================================
# Add socioeconomic reference values
# ============================================================================

mobility_metrics["employed_reference"] = (
    mobility_metrics["insee_code"]
    .map(
        {
            node: attributes.get("employed")
            for node, attributes in graph.nodes(data=True)
        }
    )
)

mobility_metrics["total_jobs_reference"] = (
    mobility_metrics["insee_code"]
    .map(
        {
            node: attributes.get("total_jobs")
            for node, attributes in graph.nodes(data=True)
        }
    )
)

In [ ]:
# ============================================================================
# Compare graph-derived and reference employment values
# ============================================================================

mobility_metrics["resident_workers_ratio"] = (
    mobility_metrics["resident_workers"]
    / mobility_metrics["employed_reference"]
)

mobility_metrics["workplace_workers_ratio"] = (
    mobility_metrics["workplace_workers"]
    / mobility_metrics["total_jobs_reference"]
)

comparison = mobility_metrics[
    [
        "resident_workers",
        "employed_reference",
        "resident_workers_ratio",
        "workplace_workers",
        "total_jobs_reference",
        "workplace_workers_ratio",
    ]
].copy()

# ---------------------------------------------------------------------------
# Replace infinite ratios caused by zero reference values
# ---------------------------------------------------------------------------

comparison = comparison.replace(
    [np.inf, -np.inf],
    np.nan,
)

comparison.describe()

In [ ]:
# Correlation matrix

mobility_metrics[
    [
        "resident_workers",
        "employed_reference",
        "workplace_workers",
        "total_jobs_reference",
    ]
].corr()

In [ ]:
# ============================================================================
# Compare resident workers with employed reference
# ============================================================================

comparison_resident = mobility_metrics[
    [
        "resident_workers",
        "employed_reference",
    ]
].dropna()

plt.figure(figsize=(7, 6))

plt.scatter(
    comparison_resident["employed_reference"],
    comparison_resident["resident_workers"],
    s=8,
    alpha=0.4,
)

plt.xlabel("Employed reference")
plt.ylabel("Resident workers from mobility graph")
plt.title("Resident workers vs employed reference")

plt.xscale("log")
plt.yscale("log")

plt.savefig(
    "../assets/figures/resident_workers_vs_employed_reference.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================================
# Compare workplace workers with total jobs reference
# ============================================================================

comparison_workplace = mobility_metrics[
    [
        "workplace_workers",
        "total_jobs_reference",
    ]
].dropna()

plt.figure(figsize=(7, 6))

plt.scatter(
    comparison_workplace["total_jobs_reference"],
    comparison_workplace["workplace_workers"],
    s=8,
    alpha=0.4,
)

plt.xlabel("Total jobs reference")
plt.ylabel("Workplace workers from mobility graph")
plt.title("Workplace workers vs total jobs reference")

plt.xscale("log")
plt.yscale("log")

plt.savefig(
    "../assets/figures/workplace_workers_vs_total_jobs_reference.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Ratio histogram

ratio_plot = mobility_metrics[
    [
        "resident_workers_ratio",
        "workplace_workers_ratio",
    ]
].replace(
    [np.inf, -np.inf],
    np.nan,
)

ratio_plot.hist(
    bins=80,
    figsize=(8, 5),
)

plt.savefig(
    "../assets/figures/mobility_reference_ratios.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================================
# Check municipality mobility coverage by department
# ============================================================================

node_reference = pd.DataFrame(
    [
        {
            "insee_code": node,
            "municipality": attributes.get("municipality"),
            "department_code": attributes.get("department_code"),
            "entity_type": attributes.get("entity_type"),
        }
        for node, attributes in graph.nodes(data=True)
    ]
)

origin_codes = set(
    edges_df["origin_insee_code"]
)

destination_codes = set(
    edges_df["destination_insee_code"]
)

node_reference["has_outgoing_flow"] = (
    node_reference["insee_code"].isin(origin_codes)
)

node_reference["has_incoming_flow"] = (
    node_reference["insee_code"].isin(destination_codes)
)

coverage_by_department = (
    node_reference[
        node_reference["entity_type"] == "COMMUNE"
    ]
    .groupby("department_code")
    .agg(
        municipalities=("insee_code", "size"),
        with_outgoing=("has_outgoing_flow", "sum"),
        with_incoming=("has_incoming_flow", "sum"),
    )
)

coverage_by_department["outgoing_coverage"] = (
    coverage_by_department["with_outgoing"]
    / coverage_by_department["municipalities"]
)

coverage_by_department["incoming_coverage"] = (
    coverage_by_department["with_incoming"]
    / coverage_by_department["municipalities"]
)

coverage_by_department.sort_values(
    "outgoing_coverage"
).head(30)